In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from IPython import display as disp

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
def cycle(iterable):
    while True:
        for x in iterable:
            yield x

class_names = ['apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle','bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel','can','castle','caterpillar','cattle','chair','chimpanzee','clock','cloud','cockroach','couch','crab','crocodile','cup','dinosaur','dolphin','elephant','flatfish','forest','fox','girl','hamster','house','kangaroo','computer_keyboard','lamp','lawn_mower','leopard','lion','lizard','lobster','man','maple_tree','motorcycle','mountain','mouse','mushroom','oak_tree','orange','orchid','otter','palm_tree','pear','pickup_truck','pine_tree','plain','plate','poppy','porcupine','possum','rabbit','raccoon','ray','road','rocket','rose','sea','seal','shark','shrew','skunk','skyscraper','snail','snake','spider','squirrel','streetcar','sunflower','sweet_pepper','table','tank','telephone','television','tiger','tractor','train','trout','tulip','turtle','wardrobe','whale','willow_tree','wolf','woman','worm',]

import torchvision.transforms as T

cifar100_mean = (0.5071, 0.4867, 0.4408)
cifar100_std  = (0.2675, 0.2565, 0.2761)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),   
    T.ToTensor(),
    T.Normalize(cifar100_mean, cifar100_std),
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(cifar100_mean, cifar100_std),
])

train_loader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR100('data', train=True, download=True, transform=train_transform),
    batch_size=128, shuffle=True, drop_last=True)

test_loader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR100('data', train=False, download=True, transform=test_transform),
    batch_size=128, shuffle=False, drop_last=False)


train_iterator = iter(cycle(train_loader))
test_iterator = iter(cycle(test_loader))

print(f'> Size of training dataset {len(train_loader.dataset)}')
print(f'> Size of test dataset {len(test_loader.dataset)}')

In [ ]:
plt.rcParams['figure.dpi'] = 70
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    img = test_loader.dataset[i][0].numpy().transpose(1, 2, 0)
    plt.imshow(img)
    plt.xlabel(class_names[test_loader.dataset[i][1]])
plt.show()

In [ ]:
class MBConvSEResBlock(nn.Module):
    """
    MobileNetV2-style MBConv block with SE and residual.
    Order: pointwise (expand) -> depthwise -> SE -> pointwise (project).
    """
    def __init__(self, in_ch, out_ch, stride=1, expansion=2, se_reduce=8):
        super().__init__()
        mid_ch = in_ch * expansion
        self.use_res = (stride == 1 and in_ch == out_ch)

        # 1x1 pointwise expand
        self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False)
        self.bn1   = nn.BatchNorm2d(mid_ch)

        # 3x3 depthwise conv
        self.dw    = nn.Conv2d(
            mid_ch, mid_ch,
            kernel_size=3, stride=stride, padding=1,
            groups=mid_ch, bias=False
        )
        self.bn2   = nn.BatchNorm2d(mid_ch)

        # SE on expanded channels
        se_mid = max(4, mid_ch // se_reduce)
        self.se1  = nn.Conv2d(mid_ch, se_mid, kernel_size=1)
        self.se2  = nn.Conv2d(se_mid, mid_ch, kernel_size=1)

        # 1x1 pointwise project
        self.conv2 = nn.Conv2d(mid_ch, out_ch, kernel_size=1, bias=False)
        self.bn3   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        identity = x

        x = F.silu(self.bn1(self.conv1(x)))
        x = F.silu(self.bn2(self.dw(x)))

        # SE
        w = x.mean((2, 3), keepdim=True)
        w = F.relu(self.se1(w))
        w = torch.sigmoid(self.se2(w))
        x = x * w

        x = self.bn3(self.conv2(x))

        if self.use_res:
            x = x + identity

        return F.silu(x)



class MBConvCIFAR100Net(nn.Module):
    """
    MBConv + SE + residual network for CIFAR-100.
    Channel schedule: 26 -> 36 -> 52 -> 72
    Total params ≈ 97,662 (under 100k).
    """
    def __init__(self, n_classes=100):
        super().__init__()

        # Stem: 3 -> 26, 32x32
        self.stem = nn.Sequential(
            nn.Conv2d(3, 26, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(26),
            nn.SiLU(),
        )

        # Stage 1: 32x32, 26 channels
        self.stage1 = nn.Sequential(
            MBConvSEResBlock(26, 26, stride=1, expansion=2),
            MBConvSEResBlock(26, 26, stride=1, expansion=2),
        )

        # Stage 2: 32x32 -> 16x16, 36 channels
        self.stage2 = nn.Sequential(
            MBConvSEResBlock(26, 36, stride=2, expansion=2),
            MBConvSEResBlock(36, 36, stride=1, expansion=2),
        )

        # Stage 3: 16x16 -> 8x8, 52 channels
        self.stage3 = nn.Sequential(
            MBConvSEResBlock(36, 52, stride=2, expansion=2),
            MBConvSEResBlock(52, 52, stride=1, expansion=2),
        )

        # Stage 4: 8x8 -> 4x4, 72 channels
        self.stage4 = nn.Sequential(
            MBConvSEResBlock(52, 72, stride=2, expansion=2),
            MBConvSEResBlock(72, 72, stride=1, expansion=2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(p=0.2)
        self.fc   = nn.Linear(72, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)



N = MBConvCIFAR100Net(n_classes=100).to(device)

num_params = len(torch.nn.utils.parameters_to_vector(N.parameters()))
print("> Number of parameters", num_params)

if num_params > 100000:
    print("> Warning: you have gone over your parameter budget and will have a grade penalty!")



In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimiser = torch.optim.SGD(
    N.parameters(),
    lr=0.2,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser,
    T_max=10000
)


In [ ]:
max_steps = 10000
eval_every = 500

steps = 0
plot_data = []

while steps < max_steps:
    N.train()
    train_losses = []
    train_accs = []

    for _ in range(eval_every):
        if steps >= max_steps:
            break

        x, t = next(train_iterator)
        x, t = x.to(device), t.to(device)

        optimiser.zero_grad()
        p = N(x)
        loss = criterion(p, t)
        loss.backward()
        optimiser.step()
        scheduler.step()

        pred = p.argmax(dim=1)
        acc = (pred == t).float().mean().item()

        train_losses.append(loss.item())
        train_accs.append(acc)

        steps += 1

    N.eval()
    test_accs = []
    with torch.no_grad():
        for x, t in test_loader:
            x, t = x.to(device), t.to(device)
            pred = N(x).argmax(dim=1)
            test_accs.append((pred == t).float().mean().item())

    train_loss_mean = np.mean(train_losses)
    train_acc_mean = np.mean(train_accs)
    train_acc_std = np.std(train_accs)
    test_acc_mean = np.mean(test_accs)
    test_acc_std = np.std(test_accs)

    print(f"steps: {steps}, "
          f"train loss: {train_loss_mean:.3f}, "
          f"train acc: {train_acc_mean:.3f}±{train_acc_std:.3f}, "
          f"test acc: {test_acc_mean:.3f}±{test_acc_std:.3f}"
    )

    plot_data.append([steps, train_acc_mean, train_acc_std, test_acc_mean, test_acc_std])

    plt.plot([x[0] for x in plot_data], [x[1] for x in plot_data], '-', color='tab:grey', label="Train accuracy")
    plt.fill_between([x[0] for x in plot_data], [x[1]-x[2] for x in plot_data], [x[1]+x[2] for x in plot_data], alpha=0.2, color='tab:grey')
    plt.plot([x[0] for x in plot_data], [x[3] for x in plot_data], '-', color='tab:purple', label="Test accuracy")
    plt.fill_between([x[0] for x in plot_data], [x[3]-x[4] for x in plot_data], [x[3]+x[4] for x in plot_data], alpha=0.2, color='tab:purple')
    plt.xlabel('Steps')
    plt.ylabel('Accuracy')
    plt.legend(loc="upper left")
    plt.show()
    disp.clear_output(wait=True)


In [ ]:
def plot_image(i, predictions_array, true_label, img):
    predictions_array, true_label, img = predictions_array[i], true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    img = np.transpose(img, (1, 2, 0))
    plt.imshow(img)

    predicted_label = np.argmax(predictions_array)
    color = '#335599' if predicted_label == true_label else '#ee4433'

    plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                  100*np.max(predictions_array),
                                  class_names[true_label]),
                                  color=color)

def plot_value_array(i, predictions_array, true_label):
    predictions_array, true_label = predictions_array[i], true_label[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    thisplot = plt.bar(range(100), predictions_array, color="#777777")
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)

    thisplot[predicted_label].set_color('#ee4433')
    thisplot[true_label].set_color('#335599')

test_images, test_labels = next(test_iterator)
test_images, test_labels = test_images.to(device), test_labels.to(device)
test_preds = torch.softmax(N(test_images), dim=1).data.cpu().numpy()
num_rows = 8
num_cols = 4
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, test_preds, test_labels.cpu().numpy(), test_images.cpu().numpy()) # Used .numpy() here
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, test_preds, test_labels.cpu().numpy())